In [24]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# Path to your Excel training data
TRAINING_XLSX = "data/refractory-meltingvsdensity.xlsx"

# Input columns must match the component labels / order used in DesignSpace
INPUT_COLS = ["Ti", "V", "Nb", "Mo", "Hf", "Ta", "W"]

# Target columns in your Excel sheet
TARGET_COLS = [
    "Melting Point (K)",
    "PROP RT Density (g/cm3)",
]

# Where to save the optimized models, scalers, plots, and optimization studies and results
OUTPUT_DIR = "ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS_PATH = os.path.join(OUTPUT_DIR, "models")
PLOTS_PATH = os.path.join(OUTPUT_DIR, "plots")
STUDIES_PATH = os.path.join(OUTPUT_DIR, "studies")
RESULTS_PATH = OUTPUT_DIR

In [25]:
# If your data is on a specific sheet, pass sheet_name="Sheet1" (or the right name).
df = pd.read_excel(TRAINING_XLSX)  # , sheet_name="Sheet1"
df.head()

,Unnamed: 0,Ti,V,Nb,Mo,Hf,Ta,W,PROP RT Density (g/cm3),Melting Point (K),Melting Point (°C)_Avg
0,500,0.05,0.000,0.575,0.100,0.0,0.225,0.050,10.936837,2893.050,2619.900
1,682,0.10,0.000,0.475,0.100,0.0,0.325,0.000,11.031867,2859.350,2586.200
2,809,0.05,0.000,0.600,0.050,0.0,0.225,0.075,11.100408,2909.375,2636.225
3,139,0.00,0.050,0.600,0.075,0.0,0.200,0.075,11.065715,2911.625,2638.475
4,766,0.05,0.025,0.550,0.125,0.0,0.250,0.000,10.702152,2848.775,2575.625


In [3]:
# Convert CTE
df['CTE (x 10^(-6) 1/K)'] = df['CTE (1/K)'] * 1e6
df.head()

,Fe,Co,Ni,Cr,V,CTE (1/K),K (W/(mK)),K/CTE,CTE (x 10^(-6) 1/K)
0,0.1,0.1,0.1,0.30,0.40,0.000009,7.447543,819403.1824,9.09
1,0.1,0.1,0.1,0.32,0.38,0.000009,7.715431,852620.8795,9.05
2,0.1,0.1,0.1,0.34,0.36,0.000009,7.956606,882843.4959,9.01
3,0.1,0.1,0.1,0.36,0.34,0.000009,8.171040,910002.9011,8.98
4,0.1,0.1,0.1,0.38,0.32,0.000009,8.359425,934130.3152,8.95


In [26]:
# Extract inputs (X) and targets (y)
X_raw = df[INPUT_COLS]
y_raw = df[TARGET_COLS]

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)

X_raw shape: (1000, 7)
y_raw shape: (1000, 2)


In [27]:
# Define models
models = {
    "Linear": LinearRegression(),
    "RFR": RandomForestRegressor()
}

In [28]:
# Define hyperparameter spaces for optimization
param_spaces = {
    "RFR": {
        "n_estimators": ('int', 10, 150),   # Number of trees
        "max_depth": ('int', 10, 150),         # Maximum depth of trees
        "min_samples_split": ('int', 2, 20), # Minimum number of samples required to split an internal node
        "min_samples_leaf": ('int', 1, 20)   # Minimum number of samples required to be at a leaf node
    }
}

In [31]:
# Import the run_workflow function from workflow.py
from core.armote_multi_output import run_workflow

# Define output labels
output_labels = ['Melting Point (K)', 'Density (g/cm3)']

# Define the folder dictionary
folder_dict = {
    'models': MODELS_PATH,
    'plots': PLOTS_PATH,
    'studies': STUDIES_PATH,
    'results': RESULTS_PATH
    }

# Run the workflow
results_df = run_workflow(X_raw, y_raw, models, param_spaces, output_labels=output_labels, folder_dict=folder_dict)

# Display the results DataFrame
print(results_df)

Setting up output folders...
✓ Models folder ready: c:\Users\trippr5118\Documents\Research\resource-allocation\ground_truth_models\TiVNbMoHfTaW-MeltingVsDensity\models
✓ Plots folder ready: c:\Users\trippr5118\Documents\Research\resource-allocation\ground_truth_models\TiVNbMoHfTaW-MeltingVsDensity\plots
✓ Studies folder ready: c:\Users\trippr5118\Documents\Research\resource-allocation\ground_truth_models\TiVNbMoHfTaW-MeltingVsDensity\studies
✓ Results folder ready: c:\Users\trippr5118\Documents\Research\resource-allocation\ground_truth_models\TiVNbMoHfTaW-MeltingVsDensity

Initializing multi-output workflow...
NOTE: Optimization uses SCALED data for fair comparison across outputs
      Reporting includes normalized metrics (RMSE/std) for fair comparison

Fitting and saving data scalers...
  Scalers saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\models

--- Starting Workflow for: Linear ---
  Checking multi-output support for LinearRegression...
  → LinearRegression support

[I 2026-03-18 13:59:16,853] A new study created in memory with name: no-name-1729b553-6018-4cc8-9df0-f7a058aa0934


  Y-Y plot saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\plots\Linear_yy_plot.png

--- Starting Workflow for: RFR ---
  Checking multi-output support for RandomForestRegressor...
  → RandomForestRegressor supports multi-output natively.


  3%|▎         | 3/100 [00:00<00:12,  7.84it/s]

[I 2026-03-18 13:59:16,966] Trial 1 finished with values: [0.25864733747431545, 0.7387065593002964] and parameters: {'n_estimators': 133, 'max_depth': 141, 'min_samples_split': 8, 'min_samples_leaf': 13}.
[I 2026-03-18 13:59:16,978] Trial 0 finished with values: [0.2259864948341539, 0.7701976787297907] and parameters: {'n_estimators': 75, 'max_depth': 81, 'min_samples_split': 7, 'min_samples_leaf': 14}.
[I 2026-03-18 13:59:17,008] Trial 2 finished with values: [0.14813049215480822, 0.8493775980439405] and parameters: {'n_estimators': 79, 'max_depth': 24, 'min_samples_split': 16, 'min_samples_leaf': 11}.
[I 2026-03-18 13:59:17,011] Trial 8 finished with values: [0.19012642100498015, 0.8064869881568825] and parameters: {'n_estimators': 45, 'max_depth': 79, 'min_samples_split': 15, 'min_samples_leaf': 10}.


...

[I 2026-03-18 13:59:18,954] Trial 99 finished with values: [0.2548809421811833, 0.7410969772683484] and parameters: {'n_estimators': 23, 'max_depth': 51, 'min_samples_split': 4, 'min_samples_leaf': 17}.
[I 2026-03-18 13:59:18,955] Trial 96 finished with values: [0.18531588722433795, 0.8112753475827941] and parameters: {'n_estimators': 36, 'max_depth': 51, 'min_samples_split': 20, 'min_samples_leaf': 7}.
[I 2026-03-18 13:59:18,980] Trial 98 finished with values: [0.17058635437264763, 0.8260939074377213] and parameters: {'n_estimators': 55, 'max_depth': 132, 'min_samples_split': 12, 'min_samples_leaf': 8}.
[I 2026-03-18 13:59:18,985] Trial 94 finished with values: [0.28032670874763516, 0.715370122884894] and parameters: {'n_estimators': 111, 'max_depth': 145, 'min_samples_split': 20, 'min_samples_leaf': 20}.
[I 2026-03-18 13:59:19,015] Trial 95 finished with values: [0.18940604963438654, 0.8072482558070231] and parameters: {'n_estimators': 129, 'max_depth': 17, 'min_samples_split': 2, 'm

c:\Users\trippr5118\Documents\Research\resource-allocation\core\armote_multi_output.py:416: ExperimentalWarning: optuna.visualization.matplotlib._pareto_front.plot_pareto_front is experimental (supported from v2.8.0). The interface can change in the future.
  ax = vis.plot_pareto_front(study, target_names=target_names)


  Final model training complete in 0.065 seconds.
  Optuna study saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\studies\RFR_study.pkl


c:\Users\trippr5118\Documents\Research\resource-allocation\core\armote_multi_output.py:425: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = vis.plot_optimization_history(
c:\Users\trippr5118\Documents\Research\resource-allocation\core\armote_multi_output.py:436: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = vis.plot_optimization_history(
c:\Users\trippr5118\Documents\Research\resource-allocation\core\armote_multi_output.py:447: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface can change in the future.
  ax = vis.plot_param_importances(study)


  Optuna plots for RFR saved in 'ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\plots' directory.
  Best model saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\models\RFR_best_model.pkl
  Y-Y plot saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\plots\RFR_yy_plot.png

--- Workflow Complete ---
✓ Results summary saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\results.csv
✓ Detailed results saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\results_detailed.txt
✓ Models saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\models
✓ Plots saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\plots
✓ Studies saved to: ground_truth_models/TiVNbMoHfTaW-MeltingVsDensity\studies
    Model  Optimization Time (s)  Retraining Time (s)  \
0  Linear                  0.000                0.000   
1     RFR                  2.164                0.065   

                                         Best Params Train R2 (Per-Output)  \
0           

In [8]:
# Path to your Excel training data
TRAINING_XLSX = "data/1000_refractory_alloy_calphad_modeled_data.xlsx"

# Input columns must match the component labels / order used in DesignSpace
INPUT_COLS = ["Ti", "V", "Nb", "Mo", "Hf", "Ta", "W"]

# Target columns in your Excel sheet
TARGET_COLS = [
    "PROP RT Therm Cond (W/(mK))",
    "Sconf",
]

# Where to save the optimized models, scalers, plots, and optimization studies and results
OUTPUT_DIR = "ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS_PATH = os.path.join(OUTPUT_DIR, "models")
PLOTS_PATH = os.path.join(OUTPUT_DIR, "plots")
STUDIES_PATH = os.path.join(OUTPUT_DIR, "studies")
RESULTS_PATH = OUTPUT_DIR

In [9]:
# If your data is on a specific sheet, pass sheet_name="Sheet1" (or the right name).
df = pd.read_excel(TRAINING_XLSX)  # , sheet_name="Sheet1"
df.head()

,Unnamed: 0,Ti,V,Nb,Mo,Hf,Ta,W,System Type,Unnamed: 0.2,...,Atomic Planar Density_Var,C_11_Avg,C_11_Var,C_12_Avg,C_12_Var,C'_Avg,C'_Var,Bulk Modulus (GPa)_Avg,Bulk Modulus (GPa)_Var,Pugh Ratio
0,500,0.05,0.000,0.575,0.100,0.0,0.225,0.050,17,77984,...,5.178468,16.650,39.770938,36.150,94.433191,-9.7500,31.358611,186.75,37.972852,3.132075
1,682,0.10,0.000,0.475,0.100,0.0,0.325,0.000,58,96328,...,5.697076,22.800,45.777287,41.100,95.001526,-9.1500,31.652843,179.75,30.289231,3.160440
2,809,0.05,0.000,0.600,0.050,0.0,0.225,0.075,17,80189,...,4.271883,11.400,34.318217,20.550,70.249181,-4.5750,22.844734,187.25,41.652581,3.195392
3,139,0.00,0.050,0.600,0.075,0.0,0.200,0.075,16,76428,...,5.118709,8.875,27.714786,36.350,96.610183,-13.7375,36.451507,190.50,38.791107,3.173678
4,766,0.05,0.025,0.550,0.125,0.0,0.250,0.000,28,93443,...,5.914301,19.775,41.875104,50.425,108.592792,-15.3250,38.319471,181.75,26.822332,3.215391


In [10]:
# Extract inputs (X) and targets (y)
X_raw = df[INPUT_COLS]
y_raw = df[TARGET_COLS]

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)

X_raw shape: (1000, 7)
y_raw shape: (1000, 2)


In [11]:
# Define models
models = {
    "Linear": LinearRegression(),
    "RFR": RandomForestRegressor()
}

In [12]:
# Define hyperparameter spaces for optimization
param_spaces = {
    "RFR": {
        "n_estimators": ('int', 10, 150),   # Number of trees
        "max_depth": ('int', 10, 150),         # Maximum depth of trees
        "min_samples_split": ('int', 2, 20), # Minimum number of samples required to split an internal node
        "min_samples_leaf": ('int', 1, 20)   # Minimum number of samples required to be at a leaf node
    }
}

In [13]:
# Import the run_workflow function from workflow.py
from core.armote_multi_output import run_workflow

# Define output labels
output_labels = ['K (W/(m·K))', 'ΔSᶜᵒⁿᶠⁱᵍ (R)']

# Define the folder dictionary
folder_dict = {
    'models': MODELS_PATH,
    'plots': PLOTS_PATH,
    'studies': STUDIES_PATH,
    'results': RESULTS_PATH
    }

# Run the workflow
results_df = run_workflow(X_raw, y_raw, models, param_spaces, output_labels=output_labels, folder_dict=folder_dict)

# Display the results DataFrame
print(results_df)

Setting up output folders...
✓ Models folder ready: c:\Users\shakt\Documents\gitrepo\resource-allocation\ground_truth_models\TiVNbMoHfTaW_Max_S_Max_TCond\models
✓ Plots folder ready: c:\Users\shakt\Documents\gitrepo\resource-allocation\ground_truth_models\TiVNbMoHfTaW_Max_S_Max_TCond\plots
✓ Studies folder ready: c:\Users\shakt\Documents\gitrepo\resource-allocation\ground_truth_models\TiVNbMoHfTaW_Max_S_Max_TCond\studies
✓ Results folder ready: c:\Users\shakt\Documents\gitrepo\resource-allocation\ground_truth_models\TiVNbMoHfTaW_Max_S_Max_TCond

Initializing multi-output workflow...
NOTE: Optimization uses SCALED data for fair comparison across outputs
      Reporting includes normalized metrics (RMSE/std) for fair comparison

Fitting and saving data scalers...
  Scalers saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\models

--- Starting Workflow for: Linear ---
  Checking multi-output support for LinearRegression...
  → LinearRegression supports multi-output natively.
  No

[I 2026-02-03 11:28:43,689] A new study created in memory with name: no-name-e71b6dff-bac5-4bfc-98b2-a4a24d7e7638


  Y-Y plot saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\plots\Linear_yy_plot.png

--- Starting Workflow for: RFR ---
  Checking multi-output support for RandomForestRegressor...
  → RandomForestRegressor supports multi-output natively.


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-02-03 11:28:44,400] Trial 0 finished with values: [0.16521056366215547, 0.8337993376652723] and parameters: {'n_estimators': 101, 'max_depth': 105, 'min_samples_split': 15, 'min_samples_leaf': 3}.
[I 2026-02-03 11:28:44,429] Trial 1 finished with values: [0.1660273508086155, 0.8336180486232445] and parameters: {'n_estimators': 147, 'max_depth': 63, 'min_samples_split': 18, 'min_samples_leaf': 3}.
[I 2026-02-03 11:28:45,462] Trial 7 finished with values: [0.11714934974549147, 0.8819880855948823] and parameters: {'n_estimators': 30, 'max_depth': 126, 'min_samples_split': 9, 'min_samples_leaf': 9}.
[I 2026-02-03 11:28:45,478] Trial 2 finished with values: [0.1651942842243716, 0.8333636254599363] and parameters: {'n_estimators': 10, 'max_depth': 95, 'min_samples_split': 19, 'min_samples_leaf': 17}.
[I 2026-02-03 11:28:46,259] Trial 11 finished with values: [0.1180223685380694, 0.881168609567459] and parameters: {'n_estimators': 145, 'max_depth': 111, 'min_samples_split': 20, 'min_s

c:\Users\shakt\Documents\gitrepo\resource-allocation\core\armote_multi_output.py:416: ExperimentalWarning: optuna.visualization.matplotlib._pareto_front.plot_pareto_front is experimental (supported from v2.8.0). The interface can change in the future.
  ax = vis.plot_pareto_front(study, target_names=target_names)
c:\Users\shakt\Documents\gitrepo\resource-allocation\core\armote_multi_output.py:425: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = vis.plot_optimization_history(
c:\Users\shakt\Documents\gitrepo\resource-allocation\core\armote_multi_output.py:436: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = vis.plot_optimization_history(
c:\Users\shakt\Documents\gitrepo\resource-allocation\core\armote_multi_ou

  Optuna plots for RFR saved in 'ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\plots' directory.
  Best model saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\models\RFR_best_model.pkl
  Y-Y plot saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\plots\RFR_yy_plot.png

--- Workflow Complete ---
✓ Results summary saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\results.csv
✓ Detailed results saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\results_detailed.txt
✓ Models saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\models
✓ Plots saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\plots
✓ Studies saved to: ground_truth_models/TiVNbMoHfTaW_Max_S_Max_TCond\studies
    Model  Optimization Time (s)  Retraining Time (s)  \
0  Linear                  0.000                0.001   
1     RFR                 13.896                0.275   

                                         Best Params Train R2 (Per-Output)  \
0                   